# 개인 프로젝트

# 패키지 설치

In [4]:
# 핵심 패키지들
%pip install langchain>=0.3.0 langchain-openai langchain-community langchain-text-splitters

# 추가 유용한 패키지들
%pip install faiss-cpu tiktoken python-dotenv

zsh:1: 0.3.0 not found
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
# 버전 확인
import langchain
from langchain_openai import ChatOpenAI

print(f"LangChain 버전: {langchain.__version__}")

# 간단한 테스트
try:
    llm = ChatOpenAI(model="gpt-4o-mini")
    response = llm.invoke("안녕하세요!")
    print("✅ 설치 완료 - 정상 작동")
    print(f"응답: {response.content}")
except Exception as e:
    print(f"❌ 설정 오류: {e}")

LangChain 버전: 1.2.0
✅ 설치 완료 - 정상 작동
응답: 안녕하세요! 어떻게 도와드릴까요?


### 주식 관련 지수 데이터 가져오기

In [6]:
TICKER="KO"

In [7]:
# 야후 파이낸스 주요 지수 
import yfinance as yf

sp500 = yf.download("^GSPC", period="1y", interval="1d")
vix = yf.download("^VIX", period="1y")
usd_krw = yf.download("KRW=X", period="6mo", interval="1d")

/var/folders/b7/ldss8_ss221921r5n_x8krs80000gn/T/ipykernel_10677/72843897.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download("^GSPC", period="1y", interval="1d")
[*********************100%***********************]  1 of 1 completed
/var/folders/b7/ldss8_ss221921r5n_x8krs80000gn/T/ipykernel_10677/72843897.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix = yf.download("^VIX", period="1y")
[*********************100%***********************]  1 of 1 completed
/var/folders/b7/ldss8_ss221921r5n_x8krs80000gn/T/ipykernel_10677/72843897.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  usd_krw = yf.download("KRW=X", period="6mo", interval="1d")
[*********************100%***********************]  1 of 1 completed


In [8]:
from fredapi import Fred

fred = Fred(api_key="7679fe95133779dd18e1cc61b0a16079")
cpi = fred.get_series("cpiaucsl")

### 기업 가치 데이터 가져오기

In [9]:
def get_valuation_metrics(symbol):
    t = yf.Ticker(symbol)
    info = t.info
    fin = t.financials
    bal = t.balance_sheet
    # cf = t.cashflow

    # fcf = cf.loc["Free Cash Flow"].iloc[0]
    revenue = fin.loc["Total Revenue"]
    op_income = fin.loc["Operating Income"].iloc[0]

    revenue_growth = (revenue.iloc[0] - revenue.iloc[1]) / revenue.iloc[1]
    operating_margin = op_income / revenue.iloc[0]

    ev_ebitda = info.get("enterpriseToEbitda")

    return {
        # "fcf": fcf,
        "revenue_growth": revenue_growth,
        "operating_margin": operating_margin,
        "ev_ebitda": ev_ebitda
    }
value_metrics = get_valuation_metrics(TICKER)

### 체인 호출

In [10]:
# 출력 스키마 정의
from pydantic import BaseModel, Field
from typing import Literal

class TradeDecision(BaseModel):
    decision: Literal["BUY", "SELL", "HOLD"] = Field(
        description="매수(BUY), 매도(SELL), 보유(HOLD) 중 하나"
    )
    confidence: float = Field(
        ge=0, le=1, description="판단 신뢰도 (0~1)"
    )
    reason: str = Field(
        description="판단 이유 (한 문장)"
    )


## 기업 가치 분석 데이터를 RAG로 입력

In [11]:
# Data Loader - 웹페이지 데이터 가져오기
from langchain_community.document_loaders import WebBaseLoader

# 야후 파이낸스 기업별 페이지
url = 'https://finance.yahoo.com/quote/{TICKER}/'
loader = WebBaseLoader(url)

# 웹페이지 텍스트 -> Documents
docs = loader.load()

print(len(docs))
print(len(docs[0].page_content))
print(docs[0].page_content[5000:6000])

USER_AGENT environment variable not set, consider setting it to identify your requests.


1
132



In [12]:
# Text Split (Documents -> small chunks: Documents)
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print(len(splits))


1


In [13]:
# Indexing (Texts -> Embedding -> Store)
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

vectorstore = Chroma.from_documents(documents=splits)

docs = vectorstore.similarity_search("{TICKER} 기업 가치")
print(len(docs))
print(docs[0].page_content)

1
Yahoo












Will be right back...
Thank you for your patience.
Our engineers are working quickly to resolve the issue.


## 벡터 스토어 활용하기

In [18]:
# 1단계 Data Loader - 웹페이지 데이터 가져오기
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 마이크로 소프트 공식 연례 보고서
url = 'https://www.sec.gov/Archives/edgar/data/317540/000031754025000025/coke-20241231.htm?utm_source=chatgpt.com'
loader = WebBaseLoader(url)
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,
    chunk_overlap=200,
    encoding_name='cl100k_base'
)
documents = text_splitter.split_documents(docs)
len(documents)

1

In [19]:
# 3. 벡터스토어에 문서 임베딩을 저장
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings':True},
)

vectorstore = FAISS.from_documents(documents,
                                   embedding = embeddings_model,
                                   distance_strategy = DistanceStrategy.COSINE  
                                  )

# 쿼리 실행

In [20]:
# 판단만 시키는 프롬프트
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
다음 지표와 재무재표 정보를 기반으로 투자가치를 분석하라
너는 보수적인 주식 트레이딩 분석가 워렌 버핏이다.
매수(BUY), 매도(SELL), 보유(HOLD) 중 하나를 판단하라.

규칙:
- 금액, 수량, 계좌 관련 판단을 하지 마라
- 출력 형식은 반드시 지정된 JSON 스키마를 따를 것
- 이유는 한글로 작성

지표:
- sp500: {sp500}
- vix : {vix}
- cpi : {cpi}
- usd_krw : {usd_krw}
- revenue_growth: {revenue_growth}
- operating_margin: {operating_margin}
- ev_ebitda: {ev_ebitda}
- 재무재표: {context}

""",
    input_variables=["sp500",  "vix", "cpi", "usd_krw", 
                     "revenue_growth", "operating_margin", "ev_ebitda","context"],
)


In [21]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

# LLM
model = ChatOpenAI(model='gpt-4o', temperature=0)

# Retrieval
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'lambda_mult': 0.15}
)

# Combine Documents
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# RAG Chain 연결
rag_chain = (
    {
        "context": itemgetter("query") | retriever | format_docs,
        "sp500": itemgetter("sp500"),
        "vix": itemgetter("vix"),
        "cpi": itemgetter("cpi"),
        "usd_krw": itemgetter("usd_krw"),
        "revenue_growth": itemgetter("revenue_growth"),
        "operating_margin": itemgetter("operating_margin"),
        "ev_ebitda": itemgetter("ev_ebitda"),
    }
    | prompt
    | model  # ← structured_output 제거
    | StrOutputParser()
)


parser = StrOutputParser()


result = rag_chain.invoke({
    "query": "투자 판단해줘",
    "sp500": sp500,
    "vix": vix,
    "cpi": cpi,
    "usd_krw": usd_krw,
    "revenue_growth": value_metrics["revenue_growth"],
    "operating_margin": value_metrics["operating_margin"],
    "ev_ebitda": value_metrics["ev_ebitda"],
})

print(result)


```json
{
  "decision": "HOLD",
  "reason": "현재 S&P 500 지수는 상승세를 보이고 있으며, 이는 시장의 긍정적인 분위기를 반영합니다. 그러나 VIX 지수가 낮은 수준을 유지하고 있어 시장의 변동성이 낮음을 나타내고 있습니다. 이는 투자자들이 시장에 대해 비교적 안정적인 시각을 가지고 있음을 시사합니다. CPI 지수는 꾸준히 상승하고 있어 인플레이션 압력이 존재할 수 있음을 나타내지만, 이는 아직 시장에 큰 영향을 미치지 않는 것으로 보입니다. 환율(USD/KRW)은 최근 변동성이 있지만, 이는 글로벌 경제 상황에 따른 일시적인 현상일 수 있습니다. 재무 지표를 보면, 매출 성장률과 영업 이익률이 양호한 수준을 유지하고 있으나, EV/EBITDA 비율이 높아 기업의 가치가 다소 고평가되어 있을 가능성이 있습니다. 따라서 현재로서는 추가적인 시장 변동성을 관찰하며 보유(HOLD)하는 것이 적절하다고 판단됩니다."
}
```
